# Healthcare Policies RAG — LangChain Workflow

Ingest → semantic chunk → vector store → retrieve → evaluate, using the Phase 2 healthcare policy corpus and evaluation set.


## 1. Setup


In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "Phase_2" / "data" / "healthcare_policies"
EVAL_PATH = NOTEBOOK_DIR / "Phase_2" / "data" / "evaluation" / "rag_evaluation_testset.json"
VECTOR_DB_DIR = NOTEBOOK_DIR / "Langchain" / "artifacts" / "chroma_healthcare_semantic"
COLLECTION_NAME = "healthcare_policies_semantic"

os.environ["ANONYMIZED_TELEMETRY"] = "False"
load_dotenv(NOTEBOOK_DIR / ".env", override=True)

embeddings = AzureOpenAIEmbeddings(
    azure_deployment=os.environ["AZURE_OPENAI_EMBEDDING_MODEL"],
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-01"),
)
llm = AzureChatOpenAI(
    azure_deployment=os.environ["AZURE_OPENAI_MODEL"],
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-01"),
    temperature=1,
)


## 2. Ingest — load policy PDFs with manifest metadata


In [2]:
manifest = json.loads((DATA_DIR / "policy_manifest.json").read_text(encoding="utf-8"))

documents = []
for row in manifest:
    for page in PyPDFLoader(str(DATA_DIR / row["filename"])).load():
        page.metadata.update(
            {k: row[k] for k in ["doc_id", "title", "plan_type", "policy_domain", "status"]}
        )
        documents.append(page)

print(f"Loaded {len(documents)} pages from {len(manifest)} policies.")


Loaded 5 pages from 5 policies.


## 3. Chunk — semantic chunking


In [3]:
splitter = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90,
)
chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} semantic chunks.")


Created 12 semantic chunks.


## 4. Vector store — embed and persist to Chroma


In [4]:
import shutil
shutil.rmtree(VECTOR_DB_DIR, ignore_errors=True)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=str(VECTOR_DB_DIR),
)

print(f"Persisted {vector_store._collection.count()} chunks to {VECTOR_DB_DIR}.")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Persisted 12 chunks to C:\Users\AP51039\Desktop\zs_ai_participants_repo\notebook\Langchain\artifacts\chroma_healthcare_semantic.


## 5. Retrieve + generate — LCEL RAG chain


In [5]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['doc_id']} p{d.metadata['page']}] {d.page_content}" for d in docs)

prompt = ChatPromptTemplate.from_template(
    """Answer using only the retrieved policy context. If the context is insufficient, say so.

    Context:
    {context}

    Question:
    {question}"""
)

rag_chain = (
    {"context": retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


## 6. Evaluate — retrieval hit-rate on the evaluation test set


In [6]:
testset = json.loads(EVAL_PATH.read_text(encoding="utf-8"))
questions = [case["question"] for case in testset]

retrieved_batches = retriever.batch(questions)

eval_rows = []
for case, hits in zip(testset, retrieved_batches):
    retrieved_doc_ids = [d.metadata["doc_id"] for d in hits]
    expected = case.get("expected_doc_id")
    eval_rows.append(
        {
            "id": case["id"],
            "question": case["question"],
            "expected_doc_id": expected,
            "retrieved_doc_ids": retrieved_doc_ids,
            "retrieval_hit": None if expected is None else expected in retrieved_doc_ids,
        }
    )

eval_df = pd.DataFrame(eval_rows)
display(eval_df)

hit_rate = eval_df["retrieval_hit"].dropna().astype(float).mean()
print(f"Retrieval hit rate: {hit_rate:.2%}")


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


,id,question,expected_doc_id,retrieved_doc_ids,retrieval_hit
0,Q01,"For a Gold PPO member, does the 11th physical ...",GOLD-PPO-2026,"[GOLD-PPO-2026, SILVER-HMO-2026, REHAB-2026, I...",True
1,Q02,"For a Silver HMO member, does the 7th physical...",SILVER-HMO-2026,"[SILVER-HMO-2026, GOLD-PPO-2026, SILVER-HMO-20...",True
2,Q03,Does emergency department MRI imaging require ...,IMG-UM-2026,"[SILVER-HMO-2026, IMG-UM-2026, GOLD-PPO-2026, ...",True
3,Q04,How many chiropractic visits are covered annua...,GOLD-PPO-2026,"[GOLD-PPO-2026, SILVER-HMO-2026, REHAB-2026, S...",True
4,Q05,How many chiropractic visits are covered annua...,SILVER-HMO-2026,"[SILVER-HMO-2026, SILVER-HMO-2026, GOLD-PPO-20...",True
5,Q06,Within how many calendar days should a provide...,CLAIMS-OPS-2026,"[CLAIMS-OPS-2026, SILVER-HMO-2026, GOLD-PPO-20...",True
6,Q07,What is the standard clean-claim filing limit ...,CLAIMS-OPS-2026,"[GOLD-PPO-2026, CLAIMS-OPS-2026, SILVER-HMO-20...",True
7,Q08,Does prior authorization guarantee that a clai...,GOLD-PPO-2026,"[SILVER-HMO-2026, GOLD-PPO-2026, GOLD-PPO-2026...",True
8,Q09,What clinical information should be included i...,IMG-UM-2026,"[IMG-UM-2026, SILVER-HMO-2026, GOLD-PPO-2026, ...",True
9,Q10,What is the annual acupuncture visit limit und...,NaN,"[GOLD-PPO-2026, SILVER-HMO-2026, REHAB-2026, G...",None


Retrieval hit rate: 100.00%


## 7. Sample generation


In [7]:
sample_question = testset[0]["question"]
print(sample_question)
print(rag_chain.invoke(sample_question))


For a Gold PPO member, does the 11th physical therapy visit require prior authorization?


Yes. For Gold PPO members, prior authorization is required beginning with the 11th physical therapy visit.
